In [9]:
%%writefile ../../scripts/2024merging.py

import pandas as pd 

Overwriting ../../scripts/2024merging.py


In [10]:
findex = pd.read_csv("findex_cleaned_timeseries.csv")
pip = pd.read_csv("pip_cleaned_timeseries.csv")

In [11]:
## standardize column names 

findex.columns = findex.columns.str.lower().str.strip()
pip.columns = pip.columns.str.lower().str.strip()

In [12]:
## ensure key columns match 

if "countrycode" in findex.columns:
    findex = findex.rename(columns = {"countrycode" : "country_code"})

if "countrycode" in pip.columns:
    pip = pip.rename(columns = {"countrycode" : "country_code"})

if "year " in pip.columns:
    pip = pip.rename(columns = {"year " : "year"})

#check 
assert "country_code" in findex.columns, "country_code missing in findex"
assert "year" in findex.columns, "year missing in findex"
assert "country_code" in pip.columns, "country_code missing in pip"
assert "year" in pip.columns, "year missing in pip"

In [13]:
## filter pip for only rows present in findex 
findex_years = findex["year"].unique()
pip_filtered = pip[pip["year"].isin(findex_years)]

print("\nYears in Findex:", sorted(findex_years))
print("Years after filtering PIP:", sorted(pip_filtered["year"].unique()))


Years in Findex: [np.int64(2014), np.int64(2017), np.int64(2021), np.int64(2022)]
Years after filtering PIP: [np.int64(2014), np.int64(2017), np.int64(2021), np.int64(2022)]


In [14]:
## merge datasets
merged = pd.merge(
    findex,
    pip_filtered,
    on=["country_code", "year"],
    how="inner"
)

print("\nMerged dataset shape:", merged.shape)
merged.head()


Merged dataset shape: (436, 9)


,country,country_code,year,account_pct,bank_account_pct,mobile_account_pct,country_name,poverty_headcount,gini
0,Argentina,ARG,2014,0.501973,0.501973,0.004323,Argentina,0.010591,0.418225
1,Armenia,ARM,2014,0.176691,0.172400,0.006578,Armenia,0.034376,0.314833
2,Bolivia,BOL,2014,0.417967,0.407108,0.027771,Bolivia,0.055982,0.478292
3,Brazil,BRA,2014,0.681232,0.681232,0.008577,Brazil,0.047357,0.520297
4,Burkina Faso,BFA,2014,0.143576,0.134245,0.030810,Burkina Faso,0.587375,0.353012


In [15]:
## check merge 
missing_summary = merged.isnull().sum()
print("\nMissing values by column:\n", missing_summary)

# Duplicate country-year rows
duplicates = merged.duplicated(subset=["country_code", "year"]).sum()
print("\nDuplicate country-year rows:", duplicates)


Missing values by column:
 country               0
country_code          0
year                  0
account_pct           0
bank_account_pct      0
mobile_account_pct    0
country_name          0
poverty_headcount     0
gini                  0
dtype: int64

Duplicate country-year rows: 350


In [16]:
## save merged dataset 
output_path = "../processed/master_timeseries_dataset.csv"
merged.to_csv(output_path, index=False)

print(f"\nMaster dataset saved to: {output_path}")



Master dataset saved to: ../processed/master_timeseries_dataset.csv


In [17]:
!jupyter nbconvert --to script merging.ipynb --output-dir='../../scripts/' --output='2024merging.py'

[NbConvertApp] Converting notebook merging.ipynb to script
[NbConvertApp] Writing 1955 bytes to ..\..\scripts\2024merging.py.py
